In [ ]:
import heapq
import math
from itertools import count
from pathlib import Path

import pandas as pd

In [ ]:
data_dirs = (Path("Datos"), Path("..") / "Datos")
DATA_DIR = next(
    (
        path
        for path in data_dirs
        if (path / "nodos_medellin.csv").is_file()
        and (path / "aristas_medellin.csv").is_file()
    ),
    None,
)
if DATA_DIR is None:
    raise FileNotFoundError("No se encontraron los CSV dentro de Datos/.")

nodos = pd.read_csv(DATA_DIR / "nodos_medellin.csv", index_col="osmid")
aristas = pd.read_csv(DATA_DIR / "aristas_medellin.csv")

nodos.index = pd.to_numeric(nodos.index, errors="raise").astype("int64")
nodos[["y", "x"]] = nodos[["y", "x"]].apply(pd.to_numeric, errors="raise")
aristas[["u", "v"]] = aristas[["u", "v"]].astype("int64")
aristas["length"] = pd.to_numeric(aristas["length"], errors="raise")

if nodos[["y", "x"]].isna().any().any():
    raise ValueError("Hay nodos sin coordenadas geográficas.")
if aristas["length"].isna().any() or (aristas["length"] < 0).any():
    raise ValueError("Las aristas deben tener longitudes no negativas.")

def construir_grafo(nodos, aristas):
    grafo = {node_id: [] for node_id in nodos.index}

    for u, v, longitud in aristas[["u", "v", "length"]].itertuples(
        index=False, name=None
    ):
        if u not in grafo or v not in grafo:
            raise ValueError(f"La arista ({u}, {v}) referencia un nodo inexistente.")
        if not math.isfinite(longitud):
            raise ValueError(f"Longitud no finita en la arista ({u}, {v}).")
        grafo[u].append((v, float(longitud)))

    return grafo

G = construir_grafo(nodos, aristas)

In [8]:
nodos

,y,x,highway,street_count,railway,geometry
osmid,,,,,,
309026271,6.211516,-75.572858,traffic_signals,3,NaN,POINT (-75.57286 6.21152)
309026273,6.210624,-75.570522,traffic_signals,4,NaN,POINT (-75.57052 6.21062)
309026274,6.209966,-75.568772,traffic_signals,4,NaN,POINT (-75.56877 6.20997)
309026275,6.209438,-75.567434,NaN,3,NaN,POINT (-75.56743 6.20944)
309026276,6.209127,-75.566649,NaN,4,NaN,POINT (-75.56665 6.20913)
...,...,...,...,...,...,...
14288086581,6.202989,-75.577162,NaN,3,NaN,POINT (-75.57716 6.20299)
14288086584,6.202900,-75.577284,NaN,4,NaN,POINT (-75.57728 6.2029)
14352566102,6.283407,-75.539434,NaN,1,NaN,POINT (-75.53943 6.28341)


In [9]:
aristas

osmid       highway lanes  \
u           v           key                                              
309026271   404910004   0                570316480     secondary     3   
309026273   393988726   0                 34939944     secondary     3   
            338171294   0                913377168       primary     3   
309026274   416048073   0               1153586518     secondary     3   
            9857457488  0    [35476637, 569672822]   residential     1   
...                                            ...           ...   ...   
14288086584 11956147502 0               1568625005  primary_link   NaN   
            14288086581 0               1568625007  primary_link   NaN   
14352566102 2643730354  0               1575857301   residential   NaN   
14352568819 1955050634  0               1575858040   residential   NaN   
14352568820 1291285098  0               1575858041   residential   NaN   

                            maxspeed                   name  oneway reversed  \
u           v           key                                                    
309026271   404910004   0         50               Calle 10    True    False   
309026273   393988726   0         50               Calle 10    True    False   
            338171294   0        NaN     Avenida El Poblado    True    False   
309026274   416048073   0         50               Calle 10    True    False   
            9857457488  0        NaN  [Calle 9, Carrera 41]    True    False   
...                              ...                    ...     ...      ...   
14288086584 11956147502 0        NaN                    NaN   False     True   
            14288086581 0        NaN                    NaN   False    False   
14352566102 2643730354  0        NaN                    NaN   False     True   
14352568819 1955050634  0        NaN                    NaN   False    False   
14352568820 1291285098  0        NaN                    NaN   False    False   

                                length  \
u           v           key              
309026271   404910004   0    42.692226   
309026273   393988726   0    12.333670   
            338171294   0    90.707233   
309026274   416048073   0     6.740813   
            9857457488  0    93.965892   
...                                ...   
14288086584 11956147502 0     6.991915   
            14288086581 0    17.202676   
14352566102 2643730354  0    21.400132   
14352568819 1955050634  0    39.282675   
14352568820 1291285098  0    53.173243   

                                                                      geometry  \
u           v           key                                                      
309026271   404910004   0     LINESTRING (-75.57286 6.21152, -75.5725 6.21138)   
309026273   393988726   0    LINESTRING (-75.57052 6.21062, -75.57042 6.21059)   
            338171294   0    LINESTRING (-75.57052 6.21062, -75.57056 6.210...   
309026274   416048073   0    LINESTRING (-75.56877 6.20997, -75.56872 6.20994)   
            9857457488  0    LINESTRING (-75.56877 6.20997, -75.5688 6.2099...   
...                                                                        ...   
14288086584 11956147502 0      LINESTRING (-75.57728 6.2029, -75.57735 6.2029)   
            14288086581 0    LINESTRING (-75.57728 6.2029, -75.57724 6.2029...   
14352566102 2643730354  0    LINESTRING (-75.53943 6.28341, -75.53947 6.283...   
14352568819 1955050634  0    LINESTRING (-75.56171 6.27468, -75.56165 6.27502)   
14352568820 1291285098  0     LINESTRING (-75.5606 6.27439, -75.56052 6.27486)   

                            width bridge tunnel junction access  ref  
u           v           key                                           
309026271   404910004   0     NaN    NaN    NaN      NaN    NaN  NaN  
309026273   393988726   0     NaN    NaN    NaN      NaN    NaN  NaN  
            338171294   0     NaN    NaN    NaN      NaN    NaN  NaN  
309026274   416048073   0     NaN    NaN    NaN      NaN    NaN  NaN  
 

In [3]:
class Node:
    def __init__(self, state, parent=None, cost=0):
        self.state = state
        self.parent = parent
        self.cost = cost

    def __repr__(self):
        return f"{self.state}(g={self.cost})"

In [4]:
def reconstruct_path(node):
    path = []

    while node is not None:
        path.append(node.state)
        node = node.parent

    return list(reversed(path))

In [ ]:
class PriorityFrontier:
    def __init__(self):
        self.heap = []
        self.counter = count()

    def add(self, node, priority):
        heapq.heappush(
            self.heap,
            (priority, next(self.counter), node)
        )

    def empty(self):
        return len(self.heap) == 0

    def remove(self):
        if self.empty():
            raise Exception("La frontier está vacía.")

        priority, _, node = heapq.heappop(self.heap)
        return node, priority

In [6]:
def uniform_cost_search(graph, start, goal, verbose=True):
    frontier = PriorityFrontier()
    frontier.add(Node(start, cost=0), priority=0)

    # Mejor costo conocido para cada estado.
    best_cost = {start: 0}

    explored = set()
    expansion_order = []

    while not frontier.empty():
        node, priority = frontier.remove()

        # Ignorar entradas antiguas de la cola de prioridad.
        if node.cost != best_cost.get(node.state):
            continue

        expansion_order.append(node.state)

        if verbose:
            print(f"Expandiendo {node.state}: g={node.cost}")

        if node.state == goal:
            return {
                "path": reconstruct_path(node),
                "expansion_order": expansion_order,
                "cost": node.cost
            }

        explored.add(node.state)

        for child_state, edge_cost in graph[node.state]:
            new_cost = node.cost + edge_cost

            if new_cost < best_cost.get(child_state, float("inf")):
                best_cost[child_state] = new_cost

                child = Node(
                    state=child_state,
                    parent=node,
                    cost=new_cost
                )

                frontier.add(child, priority=new_cost)

    return None

In [ ]:
def haversine_distance(origin, destination):
    """Distancia geodésica entre (latitud, longitud) en metros."""
    lat1, lon1 = map(math.radians, origin)
    lat2, lon2 = map(math.radians, destination)
    delta_lat = lat2 - lat1
    delta_lon = lon2 - lon1
    haversine = (
        math.sin(delta_lat / 2) ** 2
        + math.cos(lat1) * math.cos(lat2) * math.sin(delta_lon / 2) ** 2
    )
    return 2 * 6_371_000 * math.asin(math.sqrt(haversine))


def a_star_search(graph, coordinates, start, goal, verbose=False):
    if start not in graph or goal not in graph:
        raise ValueError("El nodo inicial y el nodo objetivo deben existir en el grafo.")

    def heuristic(node_id):
        return haversine_distance(coordinates[node_id], coordinates[goal])

    frontier = PriorityFrontier()
    frontier.add(Node(start, cost=0), priority=heuristic(start))

    best_cost = {start: 0}

    expansion_order = []

    while not frontier.empty():
        node, _ = frontier.remove()

        # Ignorar entradas antiguas de la cola de prioridad.
        if node.cost != best_cost.get(node.state):
            continue

        expansion_order.append(node.state)

        if verbose:
            print(f"Expandiendo {node.state}: g={node.cost}")

        if node.state == goal:
            return {
                "path": reconstruct_path(node),
                "expansion_order": expansion_order,
                "cost": node.cost
            }

        for child_state, edge_cost in graph[node.state]:
            new_cost = node.cost + edge_cost

            if new_cost < best_cost.get(child_state, float("inf")):
                best_cost[child_state] = new_cost

                child = Node(
                    state=child_state,
                    parent=node,
                    cost=new_cost
                )

                frontier.add(child, priority=new_cost + heuristic(child_state))

    return None

In [ ]:
# Cambia estos identificadores por los nodos de origen y destino deseados.
origen = 309026271
destino = 309026274
coordenadas = {
    node_id: (float(row.y), float(row.x))
    for node_id, row in nodos[["y", "x"]].iterrows()
}

resultado = a_star_search(G, coordenadas, origen, destino)
if resultado is None:
    print(f"No se encontró una ruta dirigida entre {origen} y {destino}.")
else:
    ruta = resultado["path"]
    print(f"Longitud de la ruta: {resultado['cost']:.2f} m")
    print(f"Nodos de la ruta: {len(ruta)}")
    print(f"Nodos expandidos: {len(resultado['expansion_order'])}")
    print("Ruta:", ruta)